# Объединённый этап 6–7: короткая итоговая проверка
Новых данных нет. Нужны свои признаки этапа 4 и выбранная по validation модель этапа 5.
**Перед запуском заполните пять строк протокола в README. Ноутбук открывает test.**
После просмотра test не подбирайте по нему новую модель и не объявляйте такую оценку независимой.

Укажите свои пути ниже. Код рассчитан на формат bundle стартового решения этапа 5; при собственном формате адаптируйте загрузку/схему. Baseline ниже повторяет правило этапа 5: неизвестный долг ниже известных. Если ранее согласовали другое правило, сохраните его.


In [ ]:
from pathlib import Path
import joblib
from evaluation_utils import load_holdout, predict_risk
from campaign_metrics import campaign_metrics

PREPARED = Path("../../lesson4_prepared")
MODEL = Path("../../lesson5_result/selected_model.joblib")
OUTPUT = Path("../results")
OUTPUT.mkdir(parents=True, exist_ok=True)
bundle = joblib.load(MODEL)
pipeline = bundle["pipeline"]
fraction = bundle["config"]["campaign_fraction"]
print("Кандидат этапа 5:", bundle["selected_experiment"])


In [ ]:
# Один test и один K для модели и baseline. Обучения здесь нет.
test = load_holdout(PREPARED, bundle, "test")
score = predict_risk(pipeline, test["X"])
summary, by_date, ranked = campaign_metrics(test["keys"], test["y"], score, fraction)
debt = test["X"][bundle["config"]["baseline_feature"]]
floor = debt.min() - 1 if debt.notna().any() else -1
baseline_summary, baseline_by_date, baseline_ranked = campaign_metrics(
    test["keys"], test["y"], debt.fillna(floor).to_numpy(), fraction)
assert by_date[["snapshot_at", "n_active", "K"]].equals(
    baseline_by_date[["snapshot_at", "n_active", "K"]])
for name, metrics, predictions in [("candidate", by_date, ranked),
                                    ("baseline_debt", baseline_by_date, baseline_ranked)]:
    print(name)
    print(metrics.to_string(index=False))
    predictions.to_csv(OUTPUT / f"test_{name}_predictions.csv", sep=";",
                       encoding="utf-8-sig", index=False)
    metrics.to_csv(OUTPUT / f"test_{name}_metrics.csv", sep=";",
                   encoding="utf-8-sig", index=False)


In [ ]:
# Технический повтор загрузки той же модели, без нового fit.
repeat_bundle = joblib.load(MODEL)
repeat_score = predict_risk(repeat_bundle["pipeline"], test["X"])
_, repeat_metrics, repeat_ranked = campaign_metrics(
    test["keys"], test["y"], repeat_score, fraction)
contact_columns = ["snapshot_at", "customer_id", "rank", "selected"]
assert ranked[contact_columns].equals(repeat_ranked[contact_columns])
assert by_date.equals(repeat_metrics)
print("Повтор: список контактов и метрики совпали. Отметьте это одной строкой в README.")


## Что осталось команде
1. Перенести validation из этапа 5 и test из этого расчёта в одну таблицу README. Сравнить в 1–2 предложениях.
2. Выбрать один разбор: две ошибки (FN и FP) **или** одно сегментное сравнение по общему top-K. Объяснить смысл трёх признаков, опираясь на прежний EDA либо готовую интерпретацию.
3. Заполнить короткий README: запуск, вывод и два главных ограничения. Для повторного запуска укажите реальные пути к прежним входам и модели и нужные зависимости.
4. Подготовить презентацию **8–12 слайдов**. Регламент: **12 минут выступление + 3 минуты вопросы**, всего 15 минут на команду. Полный комплект — к началу занятия **14 ноября**.

Новый расчёт важности, стресс-тест и отдельный отчёт не требуются. Дополнительные функции в evaluation_utils.py можно пропустить. Связь в EDA не является доказательством важности для модели или причины ухода.
